In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# 1. knowlege base 구성을 위한 데이터 생성

In [2]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("./tex_docx/소득세법_with_table.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n","\n"," ",""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n","\n"," ",""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 4 s
Wall time: 4.38 s


180

In [3]:
documents[43]

Document(metadata={'source': './tex_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1400만원 이하\n\n과세표준의 6퍼센트\n\n1400만원초과 5000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5000만원초과 8800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8800만원초과 1억5000만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5000만원초과 3억원 이하\n\n3,706만원 + (1억5,000만원을 초과하는 금액의 3

In [4]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True)
embedding = UpstageEmbeddings(
                model = 'embedding-query'
)

In [6]:
len(embedding.embed_query("소득세법"))

4096

In [12]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone,ServerlessSpec
#pinecone 객체 생성
pc = Pinecone()# api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-table'
dimension = len(embedding.embed_query("차원확인용"))

# 2. 인덱스 생성(해당인덱스가 없을경우)
existing_indexes=[index.name for index in pc.list_indexes()]
# print(index_name in existing_indexes)
if index_name not in existing_indexes:
    pc.create_index(
        name = index_name,
        dimension=dimension,
        metric='cosine', # 유사도 측정방식
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    # 인덱스가 준비될때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) # 1초대기
    print(f'인덱스{index_name}를 생성하였습니다.')
else:
    print(f'인덱스{index_name}이 이미 존재합니다.')

인덱스tax-index-table를 생성하였습니다.


In [13]:
index_name = 'tax-index-table'
pc.describe_index(index_name).status

{'ready': True, 'state': 'Ready'}

In [18]:
# 3. 인덱스 정보확인
print([index_name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index-table', 'tax-index-table', 'tax-index-table']
True
{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [20]:
%%time
# 4.문서를 임배딩해서 vector DB(index)에 저장
# 처음 한 번
from pinecone import Pinecone
pc=Pinecone()
# from langchain_pinecone import PineconeVectorStore
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할때 
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 998 μs


# 2. 답변 생성 전 retrieval 확인

In [21]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,k=3)# 기본값은 4

In [22]:
retrieved_docs[0]

Document(id='05648888-c777-42a4-9c8d-67f300a1fcaa', metadata={'source': './tex_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1400만원 이하\n\n과세표준의 6퍼센트\n\n1400만원초과 5000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5000만원초과 8800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8800만원초과 1억5000만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5000만원초과

In [23]:
retrieved_docs[1]

Document(id='0b6fbfff-ea8e-4605-a26b-f5b43390004a', metadata={'source': './tex_docx/소득세법_with_table.docx'}, page_content='제58조(재해손실세액공제) ① 사업자가 해당 과세기간에 천재지변이나 그 밖의 재해(이하 “재해”라 한다)로 대통령령으로 정하는 자산총액(이하 이 항에서 “자산총액”이라 한다)의 100분의 20 이상에 해당하는 자산을 상실하여 납세가 곤란하다고 인정되는 경우에는 다음 각 호의 소득세액(사업소득에 대한 소득세액을 말한다. 이하 이 조에서 같다)에 그 상실된 가액이 상실 전의 자산총액에서 차지하는 비율(이하 이 조에서 “자산상실비율”이라 한다)을 곱하여 계산한 금액(상실된 자산의 가액을 한도로 한다)을 그 세액에서 공제한다. 이 경우 자산의 가액에는 토지의 가액을 포함하지 아니한다. <개정 2020. 12. 29.>\n\n1. 재해 발생일 현재 부과되지 아니한 소득세와 부과된 소득세로서 미납된 소득세액\n\n2. 재해 발생일이 속하는 과세기간의 소득에 대한 소득세액\n\n② 제1항의 경우에 제56조, 제56조의2, 제57조 및 제57조의2에 따라 공제할 세액이 있을 때에는 이를 공제한 후의 세액을 소득세액으로 하여 제1항을 적용한다.<개정 2022. 12. 31.>\n\n③ 제1항에 따른 공제를 “재해손실세액공제”라 한다.\n\n④ 재해손실세액공제를 적용받으려는 자는 대통령령으로 정하는 바에 따라 관할 세무서장에게 신청할 수 있다.\n\n⑤ 관할 세무서장이 제4항의 신청을 받았을 때에는 그 공제할 세액을 결정하여 신청인에게 알려야 한다.\n\n⑥ 제4항의 신청이 없는 경우에도 제1항을 적용한다.\n\n⑦ 집단적으로 재해가 발생한 경우에는 대통령령으로 정하는 바에 따라 관할 세무서장이 조사결정한 자산상실비율에 따라 제1항을 적용한다.\n\n⑧ 재해손실세액공제에 관하여 필요한 사항은 대통령령으로 정한다.\n\n[전문개정 2009. 12. 31.]\n\n\

In [24]:
retriever = database.as_retriever(
    search_type='similarity',
    search_kwargs={'k':2}
)
retrieved_docs = retriever.invoke("연봉이 5천만원 직장인의 소득세는 얼마인가요?")
retrieved_docs[0]

Document(id='05648888-c777-42a4-9c8d-67f300a1fcaa', metadata={'source': './tex_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1400만원 이하\n\n과세표준의 6퍼센트\n\n1400만원초과 5000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5000만원초과 8800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8800만원초과 1억5000만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5000만원초과

In [28]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])
#print(retrieved_docs)

In [29]:
'55조' in retrieved_doc

True

# 3.답변 생성 

In [31]:
from openai import OpenAI
client = OpenAI()
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages=[
        {'role':'system','content':'당신은 최고의 한국 소득세법 전문가입니다.'},
        {
            'role':'user',
            'content':f"""-[context]를 참고해서 사용자의 질문에 10줄 이내로 답변해 주세요.
            -[context]:{retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)

In [33]:
result = response.choices[0].message.content
print(result)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 종합소득 과세표준과 세율을 적용해야 합니다.

1. 5천만원의 경우, 과세표준은 5천만원입니다.
2. 세율은 624만원 + (5,000만원 초과 금액의 24%)입니다.
3. 5천만원에서 4,400만원을 초과한 금액은 600만원입니다.
4. 따라서, 600만원의 24%는 144만원입니다.
5. 종합소득산출세액 = 624만원 + 144만원 = 768만원입니다.

따라서, 연봉 5천만원인 직장인의 소득세는 약 768만원입니다.


In [34]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = 'gpt-4.1-mini')

In [35]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model='solar-pro4',
    reasoning_effort='low' # 느리지만 더 깊게 추론하여 답변(low,medium,high)
) 

In [36]:
prompt = f"""[identity]
-당신은 최고의 한국 소득세법 전문가입니다.
-[context]를 참고하여 사용자의 질문에 답변해 주세요. [context]는 다음과 같아요
{retrieved_doc}
-질문:{query}"""

In [37]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

제공해주신 소득세법 조문(제54조의2, 제55조, 제59조 등)과 현행 세법 체계를 바탕으로, **연봉 5,000만원인 근로소득자의 소득세 계산 구조**를 단계별로 정리해 드립니다.

다만, 한국 소득세는 `총급여 → 소득공제 → 과세표준 → 산출세액 → 세액공제 → 결정세액`의 순서로 계산되며, **실제 납부세액은 본인의 부양가족 수, 4대보험료, 신용카드·의료비·교육비 등 특별소득공제/세액공제 적용 여부에 따라 크게 달라집니다.** 아래 계산은 가장 기본적인 공제만 가정한 **참고 계산식**입니다.

---
### 📌 전제 가정 (기본공제만 적용 시)
- 총급여액(연봉): `50,000,000원`
- 소득공제: 본인 기본공제 `1,500,000원` (다른 공제·부양가족 미고려)
- 과세표준: `50,000,000 - 1,500,000 = 48,500,000원`

---
### 1️⃣ 종합소득산출세액 계산 (제55조 제1항 세율 적용)
과세표준 `48,500,000원`은 `1,400만원 초과 ~ 5,000만원 이하` 구간에 해당합니다.

```
산출세액 = 84만원 + (48,500,000 - 14,000,000) × 15%
         = 840,000 + 34,500,000 × 0.15
         = 840,000 + 5,175,000
         = 6,015,000원
```
→ **종합소득산출세액: 6,015,000원**

---
### 2️⃣ 근로소득세액공제 적용 (제59조 제2항 제2호)
총급여액 `5,000만원`은 `3,300만원 초과 ~ 7,000만원 이하` 구간에 해당합니다.

```
공제액 = 74만원 - [(50,000,000 - 33,000,000) × 8/1000]
       = 740,000 - (17,000,000 × 0.008)
       = 740,000 - 136,000
       = 604,000원
```
단, 제59조 제2항 단서에 따라 **계산된 금액이 66만원보다 적으면 66만원으로 간주**하므로,
→

In [38]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage,UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임배딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model="text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')

embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# 2. vector DB load
# pc=Pinecone()
index_name='tax-index-table'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성
# (1)search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity',
                                    search_kwarge={'k':2}) # 유사도가 높은 top-3추출
# (2)search_type='mmr':
# retriever = vectorStore.as_retriever(search_type='mmr',
#                                     search_kwarge={'k':3,
#                                                   'fetch_k':10,
#                                                   'lambda_mult':0.5})# 1에 가까울수록 관련성 깊은 계산
# (3)search_type='similarity_score_threshold' :
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold',
#                                     search_kwarge={'k':3,
#                                                   'score_threshold':0.4})

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다.다음 문맥을 참고하여 질문에 답하세요. 
답을 모르면 모른다고 답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)

# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])



In [39]:
# 6. RAG 랭체인 구성(LCEL방식)
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 세율표에 따르면 연봉 5천만 원은 총급여에서 근로소득공제 등을 차감한 **과세표준**이 5천만 원 이하 구간에 해당합니다. 근로소득세액공제(제59조) 및 각종 공제가 적용되기 전, 과세표준 5천만 원 기준 산출세액은 **624만 원**(1,400만 원 초과분 3,600만 원의 15%인 540만 원 + 84만 원)입니다. 다만, 실제 납부세액은 인적공제·연금세액공제·근로소득세액공제 등에 따라 달라지므로 제공된 문맥만으로는 정확한 최종 세액을 확정할 수 없습니다.
